# Shock Guard Macro Allocator -- Backtest

Multi-pair crypto backtest using real Binance historical data (BTC/USDT, ETH/USDT, SOL/USDT, DOGE/USDT).

**Strategy overview:**
- Strategic Tier: Graduated position scaling (15-100%) with EMA crossover regime detection
- Tactical Tier: Shock Guard circuit breaker (2-of-3 signals: price drop, weak bids, ATR spike)
- OCO risk management: -5% stop-loss, +8% take-profit

**Analysis includes:** per-pair performance, Shock Guard trigger stats, position distribution, buy-and-hold comparison.

In [ ]:
import sys
import time
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from nautilus_trader.backtest.engine import BacktestEngine, BacktestEngineConfig
from nautilus_trader.config import LoggingConfig
from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.enums import AccountType, CurrencyType, OmsType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.model.objects import Currency, Money
from nautilus_trader.persistence.catalog import ParquetDataCatalog

# Ensure project paths are importable
PROJECT_ROOT = str(Path.cwd().resolve().parents[1]) if "strategies" in str(Path.cwd()) else str(Path.cwd().resolve())
# Try common locations
for candidate in [PROJECT_ROOT, str(Path(PROJECT_ROOT).parent)]:
    strategies_dir = Path(candidate) / "strategies"
    if strategies_dir.exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

nautilus_src = str(Path(PROJECT_ROOT) / "nautilus" / "src")
if nautilus_src not in sys.path:
    sys.path.insert(0, nautilus_src)

from strategies.crypto.shock_guard import ShockGuardConfig, ShockGuardStrategy

print(f"Project root: {PROJECT_ROOT}")
print("Imports OK")

## 1. Download Historical Data from Binance

Uses `BinanceDataProvider` to fetch 1-hour klines for all four pairs. Data is cached in `catalog/binance/` so subsequent runs skip the download.

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
PAIRS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "DOGEUSDT"]
VENUE = Venue("BINANCE")
CATALOG_DIR = Path(PROJECT_ROOT) / "catalog" / "binance"
STARTING_CAPITAL = Decimal("500")
INTERVAL = "1h"

# Trade sizes -- small fractions appropriate for $500 capital
TRADE_SIZES: dict[str, Decimal] = {
    "BTCUSDT": Decimal("0.00100"),   # ~$0.10 at ~100k
    "ETHUSDT": Decimal("0.0100"),    # ~$0.18 at ~1800
    "SOLUSDT": Decimal("0.10"),      # ~$0.13 at ~130
    "DOGEUSDT": Decimal("50"),       # ~$0.10 at ~0.002
}

# ---------------------------------------------------------------------------
# Download data via BinanceDataProvider
# ---------------------------------------------------------------------------
from nautilus_trading.data.providers import BinanceDataProvider

provider = BinanceDataProvider(pairs=PAIRS, interval=INTERVAL)
catalog = provider.ensure_catalog(CATALOG_DIR)

# Verify what we have
for pair in PAIRS:
    bars = catalog.bars(instrument_ids=[f"{pair}.BINANCE"])
    print(f"  {pair}: {len(bars):,} bars")

## 2. Helpers

In [ ]:
def _get_usdt_currency() -> Currency:
    """Return USDT currency, creating it if not already registered."""
    usdt = Currency.from_internal_map("USDT")
    if usdt is not None:
        return usdt
    return Currency(
        code="USDT",
        precision=2,
        iso4217=0,
        name="USDT",
        currency_type=CurrencyType.CRYPTO,
    )


def _get_instrument(catalog: ParquetDataCatalog, pair: str):
    """Retrieve a single instrument from the catalog by pair symbol."""
    instrument_id_str = f"{pair}.BINANCE"
    for inst in catalog.instruments():
        if str(inst.id) == instrument_id_str:
            return inst
    return None


def _bar_type_str(pair: str) -> str:
    """Return the BarType string for external 1-hour bars from Binance."""
    return f"{pair}.BINANCE-1-HOUR-LAST-EXTERNAL"


def _create_engine(starting_balance: Money) -> BacktestEngine:
    """Create a BacktestEngine configured for Binance crypto spot."""
    engine = BacktestEngine(
        config=BacktestEngineConfig(
            logging=LoggingConfig(log_level="ERROR"),
        ),
    )
    engine.add_venue(
        venue=VENUE,
        oms_type=OmsType.NETTING,
        account_type=AccountType.CASH,
        base_currency=None,
        starting_balances=[starting_balance],
        fee_model=None,  # uses maker/taker fees from instrument (0.1%)
    )
    return engine


def _extract_pnl_float(pnl_str: str) -> float:
    """Extract numeric value from NautilusTrader PnL strings like '-9.48 USD'."""
    return float(pnl_str.replace("USDT", "").replace("USD", "").strip())


def _run_backtest(pair: str) -> dict:
    """Run Shock Guard backtest on a single pair. Returns result dict."""
    bars = catalog.bars(instrument_ids=[f"{pair}.BINANCE"])
    instrument = _get_instrument(catalog, pair)
    if not bars or instrument is None:
        print(f"  [SKIP] No data for {pair}")
        return {}

    usdt = _get_usdt_currency()
    engine = _create_engine(Money(STARTING_CAPITAL, usdt))
    engine.add_instrument(instrument)
    engine.add_data(bars)

    bar_type = BarType.from_str(_bar_type_str(pair))
    config = ShockGuardConfig(
        instrument_id=instrument.id,
        bar_type=bar_type,
        trade_size=TRADE_SIZES.get(pair, Decimal("0.001")),
        default_allocation_pct=0.50,
        min_allocation_pct=0.15,
        max_allocation_pct=1.0,
        max_position_change_pct=0.25,
        regime_confidence_threshold=0.7,
        strategic_rebalance_bars=60,
        price_drop_threshold_pct=0.03,
        price_drop_window_bars=5,
        atr_spike_multiplier=5.0,
        shock_guard_signals_required=2,
        shock_guard_target_pct=0.25,
        cooldown_bars=30,
        stop_loss_pct=0.05,
        take_profit_pct=0.08,
        use_deterministic_signals=True,
        ema_fast_period=12,
        ema_slow_period=26,
    )

    strategy = ShockGuardStrategy(config=config)
    engine.add_strategy(strategy)

    t0 = time.time()
    engine.run()
    elapsed = time.time() - t0

    # Collect results
    account = engine.cache.account_for_venue(VENUE)
    balances = account.balances() if account else {}

    positions_report = engine.trader.generate_positions_report()
    fills_report = engine.trader.generate_order_fills_report()

    # Extract bar prices for buy-and-hold comparison
    bar_prices = [float(b.close) for b in bars]
    bnh_return = (bar_prices[-1] - bar_prices[0]) / bar_prices[0] if bar_prices else 0.0

    # PnL from positions
    pnl_values = []
    if not positions_report.empty and "realized_pnl" in positions_report.columns:
        pnl_values = (
            positions_report["realized_pnl"]
            .str.replace(r"\s+\w+$", "", regex=True)
            .astype(float)
            .tolist()
        )

    total_pnl = sum(pnl_values) if pnl_values else 0.0
    wins = sum(1 for p in pnl_values if p > 0)
    losses = sum(1 for p in pnl_values if p <= 0)
    win_rate = wins / len(pnl_values) if pnl_values else 0.0
    total_return = total_pnl / float(STARTING_CAPITAL)

    # Sharpe ratio (annualized, assuming 1h bars ~ 8760 bars/year)
    if pnl_values and len(pnl_values) > 1:
        pnl_arr = np.array(pnl_values)
        sharpe = (pnl_arr.mean() / pnl_arr.std()) * np.sqrt(8760 / len(bars) * len(pnl_values)) if pnl_arr.std() > 0 else 0.0
    else:
        sharpe = 0.0

    # Max drawdown from cumulative PnL
    if pnl_values:
        cum_pnl = np.cumsum(pnl_values)
        running_max = np.maximum.accumulate(cum_pnl)
        drawdowns = cum_pnl - running_max
        max_dd = float(np.min(drawdowns))
    else:
        max_dd = 0.0

    result = {
        "pair": pair,
        "bars": len(bars),
        "elapsed_s": elapsed,
        "total_pnl": total_pnl,
        "total_return": total_return,
        "win_rate": win_rate,
        "wins": wins,
        "losses": losses,
        "trades": len(pnl_values),
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "bnh_return": bnh_return,
        "balances": balances,
        "pnl_values": pnl_values,
        "bar_prices": bar_prices,
        "positions_report": positions_report,
        "fills_report": fills_report,
        "engine": engine,
        "strategy": strategy,
    }

    return result


print("Helpers defined")

## 3. Run Backtest Across All Pairs

BTC/USDT is the primary pair; ETH/USDT, SOL/USDT, and DOGE/USDT are cross-validation.

In [ ]:
results: dict[str, dict] = {}

for pair in PAIRS:
    print(f"\n{'='*50}")
    print(f"Running Shock Guard on {pair}...")
    print(f"{'='*50}")

    r = _run_backtest(pair)
    if r:
        results[pair] = r
        print(f"  Bars:          {r['bars']:,}")
        print(f"  Trades:        {r['trades']}")
        print(f"  Win rate:      {r['win_rate']:.1%}")
        print(f"  Total PnL:     {r['total_pnl']:.2f} USDT")
        print(f"  Total return:  {r['total_return']:.2%}")
        print(f"  Sharpe ratio:  {r['sharpe']:.2f}")
        print(f"  Max drawdown:  {r['max_drawdown']:.2f} USDT")
        print(f"  Buy & hold:    {r['bnh_return']:.2%}")
        print(f"  Time:          {r['elapsed_s']:.1f}s")
        print(f"  Balances:      {r['balances']}")

print(f"\n{'='*50}")
print(f"All backtests complete: {len(results)}/{len(PAIRS)} pairs")
print(f"{'='*50}")

## 4. Summary Comparison Table

In [ ]:
summary_rows = []
for pair, r in results.items():
    summary_rows.append({
        "Pair": pair,
        "Bars": r["bars"],
        "Trades": r["trades"],
        "Win Rate": f"{r['win_rate']:.1%}",
        "Total PnL (USDT)": f"{r['total_pnl']:.2f}",
        "Return": f"{r['total_return']:.2%}",
        "Sharpe": f"{r['sharpe']:.2f}",
        "Max DD (USDT)": f"{r['max_drawdown']:.2f}",
        "Buy&Hold Return": f"{r['bnh_return']:.2%}",
        "Alpha": f"{r['total_return'] - r['bnh_return']:+.2%}",
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

## 5. Shock Guard Trigger Analysis

Inspect how often the circuit breaker fired, time in market vs flat, and allocation distribution.

In [ ]:
# Shock Guard analysis: count triggers from engine logs and strategy state
# Since we have access to the strategy objects, inspect cooldown/allocation state
# We can also estimate triggers from fill patterns

for pair, r in results.items():
    engine = r["engine"]
    strategy = r["strategy"]
    fills = r["fills_report"]
    positions = r["positions_report"]

    # Count fills to estimate trading activity
    total_orders = engine.cache.orders_total_count()
    open_pos = engine.cache.positions_open_count()
    closed_pos = engine.cache.positions_closed_count()

    # Time in market: estimate from positions report
    n_bars = r["bars"]
    n_trades = r["trades"]

    # Rough estimate: each trade lasts ~rebalance_interval bars on average
    # Better: look at position open/close timestamps
    if not positions.empty and "duration_ns" in positions.columns:
        durations_ns = positions["duration_ns"].astype(float)
        total_duration_ns = durations_ns.sum()
        total_period_ns = n_bars * 3600 * 1e9  # 1h bars -> ns
        time_in_market = total_duration_ns / total_period_ns if total_period_ns > 0 else 0
    else:
        time_in_market = n_trades / max(n_bars, 1)  # rough fallback

    print(f"\n--- {pair} ---")
    print(f"  Total orders:      {total_orders}")
    print(f"  Open positions:    {open_pos}")
    print(f"  Closed positions:  {closed_pos}")
    print(f"  Time in market:    {min(time_in_market, 1.0):.1%}")
    print(f"  Time flat:         {max(1.0 - time_in_market, 0.0):.1%}")
    print(f"  Final allocation:  {strategy.current_allocation_pct:.0%}")
    print(f"  Cooldown active:   {'Yes' if strategy._cooldown_remaining > 0 else 'No'}")

## 6. Equity Curves (Strategy vs Buy-and-Hold)

In [ ]:
# Equity curves: strategy cumulative PnL vs buy-and-hold
n_pairs = len(results)
if n_pairs > 0:
    fig = make_subplots(
        rows=n_pairs, cols=1,
        subplot_titles=[f"{pair}" for pair in results],
        shared_xaxes=False,
        vertical_spacing=0.08,
    )

    colors = {"BTCUSDT": "#F7931A", "ETHUSDT": "#627EEA", "SOLUSDT": "#9945FF", "DOGEUSDT": "#C3A634"}

    for i, (pair, r) in enumerate(results.items(), 1):
        color = colors.get(pair, "cyan")

        # Strategy equity curve
        if r["pnl_values"]:
            cum_pnl = np.cumsum(r["pnl_values"])
            fig.add_trace(
                go.Scatter(
                    y=float(STARTING_CAPITAL) + cum_pnl,
                    mode="lines",
                    name=f"{pair} Strategy",
                    line=dict(color=color, width=2),
                    legendgroup=pair,
                ),
                row=i, col=1,
            )

        # Buy-and-hold equity curve (normalized to same starting capital)
        prices = np.array(r["bar_prices"])
        if len(prices) > 0:
            bnh_equity = float(STARTING_CAPITAL) * (prices / prices[0])
            # Subsample to ~500 points for plotting efficiency
            step = max(1, len(bnh_equity) // 500)
            fig.add_trace(
                go.Scatter(
                    y=bnh_equity[::step],
                    mode="lines",
                    name=f"{pair} Buy&Hold",
                    line=dict(color=color, width=1, dash="dash"),
                    legendgroup=pair,
                    opacity=0.6,
                ),
                row=i, col=1,
            )

        fig.update_yaxes(title_text="Portfolio (USDT)", row=i, col=1)

    fig.update_layout(
        title="Shock Guard vs Buy-and-Hold",
        template="plotly_dark",
        height=300 * n_pairs,
        showlegend=True,
    )
    fig.show()
else:
    print("No results to plot")

## 7. PnL Distribution per Pair

In [ ]:
# PnL distribution histograms
pairs_with_pnl = {p: r for p, r in results.items() if r["pnl_values"]}
if pairs_with_pnl:
    n = len(pairs_with_pnl)
    fig = make_subplots(rows=1, cols=n, subplot_titles=list(pairs_with_pnl.keys()))

    colors = {"BTCUSDT": "#F7931A", "ETHUSDT": "#627EEA", "SOLUSDT": "#9945FF", "DOGEUSDT": "#C3A634"}

    for i, (pair, r) in enumerate(pairs_with_pnl.items(), 1):
        fig.add_trace(
            go.Histogram(
                x=r["pnl_values"],
                nbinsx=30,
                marker_color=colors.get(pair, "cyan"),
                opacity=0.75,
                name=pair,
            ),
            row=1, col=i,
        )
        fig.update_xaxes(title_text="PnL (USDT)", row=1, col=i)
        fig.update_yaxes(title_text="Count", row=1, col=i)

    fig.update_layout(
        title="PnL Distribution per Trade",
        template="plotly_dark",
        height=400,
        showlegend=False,
    )
    fig.show()
else:
    print("No PnL data to plot")

## 8. Drawdown Analysis

In [ ]:
# Drawdown curves
pairs_with_pnl = {p: r for p, r in results.items() if r["pnl_values"]}
if pairs_with_pnl:
    fig = go.Figure()
    colors = {"BTCUSDT": "#F7931A", "ETHUSDT": "#627EEA", "SOLUSDT": "#9945FF", "DOGEUSDT": "#C3A634"}

    for pair, r in pairs_with_pnl.items():
        cum_pnl = np.cumsum(r["pnl_values"])
        running_max = np.maximum.accumulate(cum_pnl)
        drawdown = cum_pnl - running_max

        fig.add_trace(go.Scatter(
            y=drawdown,
            mode="lines",
            name=pair,
            line=dict(color=colors.get(pair, "cyan"), width=1.5),
            fill="tozeroy",
            opacity=0.5,
        ))

    fig.update_layout(
        title="Drawdown by Trade (Strategy PnL)",
        xaxis_title="Trade #",
        yaxis_title="Drawdown (USDT)",
        template="plotly_dark",
        height=400,
    )
    fig.show()
else:
    print("No PnL data to plot")

## 9. Return Comparison: Strategy vs Buy-and-Hold

In [ ]:
# Bar chart: strategy return vs buy-and-hold per pair
if results:
    pairs_list = list(results.keys())
    strat_returns = [results[p]["total_return"] * 100 for p in pairs_list]
    bnh_returns = [results[p]["bnh_return"] * 100 for p in pairs_list]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=pairs_list,
        y=strat_returns,
        name="Shock Guard",
        marker_color="#00D4AA",
    ))
    fig.add_trace(go.Bar(
        x=pairs_list,
        y=bnh_returns,
        name="Buy & Hold",
        marker_color="#FF6B6B",
    ))

    fig.update_layout(
        title="Return Comparison: Shock Guard vs Buy-and-Hold",
        xaxis_title="Pair",
        yaxis_title="Return (%)",
        barmode="group",
        template="plotly_dark",
        height=450,
    )
    fig.show()
else:
    print("No results")

## 10. Regime Testing: Price Action Overlay

Visualize the price data with the regime periods the data naturally covers (bull, bear, sideways, flash crash events).

In [ ]:
# Price action with regime detection (simple rolling return classification)
for pair in ["BTCUSDT"]:  # Focus on primary pair
    if pair not in results:
        continue
    r = results[pair]
    prices = np.array(r["bar_prices"])
    if len(prices) < 50:
        continue

    # Classify regimes using 24-bar (24h) rolling return
    window = 24
    rolling_ret = pd.Series(prices).pct_change(window).fillna(0).values

    # Classify: >2% = bull, <-2% = bear, else sideways
    regime = np.where(rolling_ret > 0.02, "Bull",
             np.where(rolling_ret < -0.02, "Bear", "Sideways"))

    # Detect flash crash candidates: >3% drop in 5 bars
    flash_crashes = []
    for j in range(5, len(prices)):
        local_max = max(prices[j-5:j])
        if local_max > 0 and (local_max - prices[j]) / local_max > 0.03:
            flash_crashes.append(j)

    # Subsample for plotting
    step = max(1, len(prices) // 1000)
    x = np.arange(len(prices))[::step]
    p = prices[::step]
    reg = regime[::step]

    fig = go.Figure()

    # Color-coded price by regime
    for reg_name, color in [("Bull", "#00FF88"), ("Bear", "#FF4444"), ("Sideways", "#888888")]:
        mask = reg == reg_name
        fig.add_trace(go.Scatter(
            x=x[mask], y=p[mask],
            mode="markers",
            marker=dict(size=2, color=color),
            name=reg_name,
        ))

    # Mark flash crash bars
    if flash_crashes:
        fc_sub = [fc for fc in flash_crashes[::max(1, len(flash_crashes)//50)]]
        fig.add_trace(go.Scatter(
            x=fc_sub,
            y=[prices[j] for j in fc_sub],
            mode="markers",
            marker=dict(size=8, color="red", symbol="triangle-down"),
            name="Flash Crash Signal",
        ))

    fig.update_layout(
        title=f"{pair} Price with Regime Classification (24h rolling)",
        xaxis_title="Bar #",
        yaxis_title="Price (USDT)",
        template="plotly_dark",
        height=500,
    )
    fig.show()

    # Summary
    total = len(regime)
    print(f"\n{pair} Regime breakdown ({total} bars):")
    for reg_name in ["Bull", "Bear", "Sideways"]:
        count = np.sum(regime == reg_name)
        print(f"  {reg_name:10s}: {count:,} bars ({count/total:.1%})")
    print(f"  Flash crash signals: {len(flash_crashes)}")

## 11. Position Reports (detailed)

In [ ]:
# Show position report for primary pair
primary = "BTCUSDT"
if primary in results and not results[primary]["positions_report"].empty:
    print(f"Positions report for {primary} (first 20):")
    display(results[primary]["positions_report"].head(20))
else:
    print(f"No positions for {primary}")

## 12. Cleanup

Dispose all engines to free resources. Go back to section 3 to re-run with different parameters.

In [ ]:
for pair, r in results.items():
    r["engine"].dispose()
    print(f"  {pair} engine disposed")

print("\nAll engines disposed -- tweak parameters in section 2 and re-run from section 3")